In [1]:
# Imports 
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter

In [2]:
# auto reload the notebook
%load_ext autoreload
%autoreload 2

In [3]:
# Directory 
BASE_DIR = Path.cwd() 


In [4]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [6]:
#### log initialization
from utils.logs import logger_module 
logger = logger_module(logger_name=__name__) 

In [7]:
# import attacks
from attack_functionality.general_attack import GeneralAttack 
attack = GeneralAttack()

In [8]:
# Load Dataset
data = pd.read_csv(BASE_DIR.parent/'data/ds_with_sample_name.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [9]:
#Train Surrogate Model
X = data.iloc[:, 2:-1].values
y = data.iloc[:, -1].values
original_indices = np.arange(len(X)) 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, original_indices, test_size=0.2, random_state=42)
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n {cm}") 
cr = classification_report(y_test, y_pred)
print(f"Classification report:\n {cr}")


Accuracy: 97.60%
Confusion Matrix:
 [[104  13]
 [  1 465]]
Classification report:
               precision    recall  f1-score   support

           0       0.99      0.89      0.94       117
           1       0.97      1.00      0.99       466

    accuracy                           0.98       583
   macro avg       0.98      0.94      0.96       583
weighted avg       0.98      0.98      0.98       583



In [10]:
idx_loaded = np.load(BASE_DIR.parent/"Adversarial_candidates_indices/dt_adversarial_candidates_indices.npy")
xtrain_poison_candidates = X_train[idx_loaded] 
print(len(xtrain_poison_candidates))

580


In [11]:
print(len(xtrain_poison_candidates))

580


In [ ]:
#Define perturbation value
perturbation=90
#Generate adversarial samples --> logs have more details
asamples, adv_idx = attack.generate_adversarial_sample(idx_loaded, xtrain_poison_candidates, perturbation, classifier)

The average  l2 distance 126.51289981624416 across 355 samples
The biggest  l2 distance 175.59328005365126 across 355 samples
The smallest l2 distance 86.98850498772812 across 355 samples
The average amount of features changed: 28.76338028169014
The biggest amount of features changed: 37
The least amount of features changed: 18


In [ ]:
# Successful Adversarial samples
len(asamples)   

355

In [ ]:
#AGR Metrics
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}")

AGR: 0.6121


In [ ]:
#if you want to save generated adversarial samples
#np.save("rf_adversarial_samples_all_features_run_2.npy", np.array(asamples))

In [19]:
### Adversarial Retrain ###  
mask = np.ones(X_train.shape[0], dtype=bool)
mask[adv_idx] = False

X_train = X_train[mask]
y_train = y_train[mask]

print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain with successful base adversarial samples removed: 1975
20% of the adversarial samples held out for testing: 71
71
199
Total length of poisoned Xtrain: 2174
199
Total length of poisoned Ytrain: 2174
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [ ]:
## Surrogate model: Adversarial Retraining ##
poisoned_classifier = RandomForestClassifier(n_estimators=100,random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test)
poisoned_accuracy = accuracy_score(y_test, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 97.77%
Confusion Matrix trained on poisoned samples:
 [[105  12]
 [  1 465]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.99      0.90      0.94       117
           1       0.97      1.00      0.99       466

    accuracy                           0.98       583
   macro avg       0.98      0.95      0.96       583
weighted avg       0.98      0.98      0.98       583



In [ ]:
## Evaluate retrained surrogate model
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [22]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [23]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Decision Tree ####
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_score(y_test, y_pred_dt):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred_xgb = model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=1.0, random_state=42)
svm_classifier.fit(X_train, y_train)
y_pred_svm_poly = svm_classifier.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred_svm_poly):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42)
svm_classifier_linear.fit(X_train, y_train)
y_pred_svm_linear = svm_classifier_linear.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred_svm_linear):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train, y_train)
ypred_lr = LR_model.predict(X_test)
accuracy = accuracy_score(y_test, ypred_lr)
print(f"accuracy of the Logistic Regression Model: {round(accuracy,2)}")

Accuracy of Decision Tree Target Model: 0.97
Accuracy of the XGboost Model: 97.60%
Accuracy of the SVM Classifier (Kernel=Poly): 0.80
Accuracy of the SVM Classifier (Kernel=Linear) : 0.97
accuracy of the Logistic Regression Model: 0.95


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [24]:
def evasion_attack(adversarial_samples,
                   DT_Model,
                   XGB_Model, 
                   SVM_poly_model, 
                   SVM_linear_model, 
                   Logistic_reg_model):
    
    predictions = {
        "Decision_Tree":[],
        "XGB":[],
        "SVM_poly": [], 
        "SVM_Linear":[], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Decision_Tree": DT_Model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model, 
        "SVM_Linear":SVM_linear_model, 
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [25]:
res = evasion_attack(asamples, classifier_dt, model, svm_classifier, svm_classifier_linear, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names)) 


 Decision_Tree: predictions

 Decision_Tree evasion rate: 284/355 samples evaded

 evaded samples:  Counter({'None': 24, 'UpdateInformation.html': 16, 'UpdateRecords.html': 15, 'Azov': 13, 'DarkSide': 12, 'Conti': 10, 'FormAttached.html': 9, 'FormUpdate.html': 8, 'Attached.html': 6, 'Nitro': 6, 'FormAttachment.html': 6, 'UpdateNow.html': 6, 'UpdateForm.html': 5, 'FORM.html': 5, 'RedLineStealer': 5, 'Attachment.html': 5, 'HiddenTear': 5, 'Egregor': 3, 'Makop': 3, 'NominatusStorm': 3, 'Stop': 3, 'Thanos': 2, 'AXLocker': 2, 'Sodinokibi': 2, 'JigsawLocker': 2, 'WannaCry': 2, 'InfinityLock': 2, 'Downloader-CUZ': 2, 'BlackCat': 2, 'GandCrab': 2, 'Flame': 2, 'BlackBasta': 2, 'Attachment764.html': 2, 'Voidcrypt': 1, 'Attachment402.html': 1, 'Form229.html': 1, 'UIWIX': 1, 'Attachment280.html': 1, 'Geodo': 1, 'CryptoDarkRubix': 1, 'Expiro': 1, 'RubberDucky': 1, 'Phobos': 1, 'Form396.html': 1, 'Hermes': 1, 'Update990.html': 1, 'Attachment560.html': 1, 'Python': 1, 'Form441.html': 1, 'UpdateForm8

In [26]:
### model transfer scaled ###

In [27]:
scaler = StandardScaler() 

Xtrain_scaled = scaler.fit_transform(X_train)
Xtest_scaled = scaler.transform(X_test)

asamples_scaled = scaler.transform(asamples)

In [28]:
### Scaling 
### Scaled Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Decision Tree ####
classifier_dt_scaled = DecisionTreeClassifier(random_state=42)
classifier_dt_scaled.fit(Xtrain_scaled, y_train)
y_pred_dt_scaled = classifier_dt_scaled.predict(Xtest_scaled)
accuracy_dt_scaled = accuracy_score(y_test, y_pred_dt_scaled) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_dt_scaled:.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model_scaled = XGBClassifier(**params)
model_scaled.fit(Xtrain_scaled, y_train)
y_pred_xgb_scaled = model_scaled.predict(Xtest_scaled)
accuracy_xgb_scaled = accuracy_score(y_test, y_pred_xgb_scaled)
print(f'Accuracy of the XGboost Model: {accuracy_xgb_scaled * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier_scaled = SVC(kernel='poly', C=1.0, random_state=42)
svm_classifier_scaled.fit(Xtrain_scaled, y_train)
y_pred_SVM_poly_scaled = svm_classifier_scaled.predict(Xtest_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred_SVM_poly_scaled):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear_scaled = SVC(kernel='linear', C=4.0, random_state=42)
svm_classifier_linear_scaled.fit(Xtrain_scaled, y_train)
y_pred_SVM_linear_scaled = svm_classifier_linear_scaled.predict(Xtest_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred_SVM_linear_scaled):.2f}")

### Logistic Regression ###
LR_model_scaled = LogisticRegression(solver='saga',max_iter=5000)
LR_model_scaled.fit(Xtrain_scaled, y_train)
ypred_lr_scaled = LR_model_scaled.predict(Xtest_scaled)
accuracy_lr_scaled = accuracy_score(y_test, ypred_lr_scaled)
print(f"accuracy of the Logistic Regression Model: {round(accuracy_lr_scaled,2)}")

Accuracy of Decision Tree Target Model: 0.97
Accuracy of the XGboost Model: 97.60%
Accuracy of the SVM Classifier (Kernel=Poly): 0.80
Accuracy of the SVM Classifier (Kernel=Linear) : 0.97
accuracy of the Logistic Regression Model: 0.97


In [29]:
def evasion_attack_scaled(adversarial_samples_scaled,
                   DT_Model_scaled,
                   XGB_Model_scaled, 
                   SVM_poly_model_scaled, 
                   SVM_linear_model_scaled, 
                   Logistic_reg_model_scaled):
    
    predictions = {
        "DT_Model_scaled":[],
        "XGB_scaled":[],
        "SVM_poly_scaled": [], 
        "SVM_Linear_scaled":[], 
        "Logistic_Regression_scaled":[]   
    }
    
    models = {
        "DT_Model_scaled": DT_Model_scaled,
        "XGB_scaled":XGB_Model_scaled,
        "SVM_poly_scaled": SVM_poly_model_scaled, 
        "SVM_Linear_scaled":SVM_linear_model_scaled, 
        "Logistic_Regression_scaled":Logistic_reg_model_scaled
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples_scaled)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [30]:

res_scaled = evasion_attack_scaled(asamples_scaled, classifier_dt_scaled, model_scaled, svm_classifier_scaled, svm_classifier_linear_scaled, LR_model_scaled)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res_scaled.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names)) 


 DT_Model_scaled: predictions

 DT_Model_scaled evasion rate: 284/355 samples evaded

 evaded samples:  Counter({'None': 24, 'UpdateInformation.html': 16, 'UpdateRecords.html': 15, 'Azov': 13, 'DarkSide': 12, 'Conti': 10, 'FormAttached.html': 9, 'FormUpdate.html': 8, 'Attached.html': 6, 'Nitro': 6, 'FormAttachment.html': 6, 'UpdateNow.html': 6, 'UpdateForm.html': 5, 'FORM.html': 5, 'RedLineStealer': 5, 'Attachment.html': 5, 'HiddenTear': 5, 'Egregor': 3, 'Makop': 3, 'NominatusStorm': 3, 'Stop': 3, 'Thanos': 2, 'AXLocker': 2, 'Sodinokibi': 2, 'JigsawLocker': 2, 'WannaCry': 2, 'InfinityLock': 2, 'Downloader-CUZ': 2, 'BlackCat': 2, 'GandCrab': 2, 'Flame': 2, 'BlackBasta': 2, 'Attachment764.html': 2, 'Voidcrypt': 1, 'Attachment402.html': 1, 'Form229.html': 1, 'UIWIX': 1, 'Attachment280.html': 1, 'Geodo': 1, 'CryptoDarkRubix': 1, 'Expiro': 1, 'RubberDucky': 1, 'Phobos': 1, 'Form396.html': 1, 'Hermes': 1, 'Update990.html': 1, 'Attachment560.html': 1, 'Python': 1, 'Form441.html': 1, 'UpdateF